# DPDA Simulator
A step-by-step simulator for Deterministic Pushdown Automata.

$$M = (Q, \Sigma, \Gamma, \delta, q_0, Z_0, F)$$

---

## Imports

In [2]:
from dataclasses import dataclass

## Stack & DPDA
The `Stack` class uses a list where the **last element is the top**.  
The `DPDA` class holds the full machine definition and exposes:
- `get_transition(state, input, stack_top)` — looks up the applicable transition (λ-transition takes priority)
- `apply_transition(stack, transition)` — pops top and pushes new symbols
- `describe_action(transition)` — returns a human-readable description of the stack operation

In [3]:
from dataclasses import dataclass


class Stack:
    def __init__(self, initial_symbol):
        self._data = [initial_symbol]

    def top(self):
        return self._data[-1] if self._data else None

    def pop(self):
        return self._data.pop() if self._data else None

    def push(self, symbol):
        self._data.append(symbol)

    def is_empty(self):
        return len(self._data) == 0

    def __str__(self):
        return ''.join(reversed(self._data)) if self._data else '(empty)'

    def __len__(self):
        return len(self._data)

    def copy(self):
        new = Stack.__new__(Stack)
        new._data = self._data.copy()
        return new





@dataclass
class DPDATransition:
    from_state:str
    input  : str  
    stack_top: str  
    to_state : str
    push:str  

    def __repr__(self):
        inp  = 'λ' if self.input == 'eps' else f"'{self.input}'"
        push = 'ε' if self.push  == 'eps' else f"'{self.push}'"
        return (f"δ({self.from_state}, {inp}, {self.stack_top}) "f"-> ({self.to_state}, {push})")


class DPDA:
    """
M = (Q, Σ, Γ, δ, q0, Z0, F)
Attributes:
    states (Q)
    input_alphabet (Σ)
    stack_alphabet(Γ)
    start_state (q0)
    initial_stack(Z0)
    final_states (F)
    acceptance_mode:'final' or 'empty'
    transitions: list of DPDATransition
    delta : dict(from_state, input, stack_top) -> DPDATransition
    """

    def __init__(self, dpda_dict, transitions):
        self.states= dpda_dict['states']
        self.input_alphabet= dpda_dict['input_alphabet']
        self.stack_alphabet= dpda_dict['stack_alphabet']
        self.start_state= dpda_dict['start_state']
        self.initial_stack= dpda_dict['initial_stack']
        self.final_states= dpda_dict['final_states']
        self.acceptance_mode= dpda_dict['acceptance_mode']
        self.transitions = transitions

        self.delta: dict[tuple, DPDATransition] =   {}
        for t in self.transitions:
            key =(t.from_state,t.input, t.stack_top)
            self.delta[key]=t

    def get_transition(self, state, input_sym, stack_top):
        eps_key =(state, 'eps', stack_top)
        if eps_key in self.delta:
            return self.delta[eps_key]

        sym_key = (state, input_sym, stack_top)
        return self.delta.get(sym_key, None)


    def apply_transition(self, stack, transition):
        new_stack = stack.copy()
        new_stack.pop()

        if transition.push !='eps':
            for symbol in reversed(transition.push):
                new_stack.push(symbol)
        return new_stack

    def describe_action(self, transition):
        top  = transition.stack_top
        push = transition.push

        if push == 'eps':
            return f"pop {top}"
        if push == top:
            return f"keep {top}"
        if len(push) > len(top):
            return f"push {push[0]}"
        return f"replace {top} with {push}"

    def is_accepting_final(self, state):
        return state in self.final_states


    def is_accepting_empty(self, stack):
        return len(stack) ==0



    def fresh_stack(self):
        return Stack(self.initial_stack)




    def stack_str(self, stack):
        return ''.join(stack) if stack else '(empty)'
    def __repr__(self):
        return (
            f"DPDA(\n"
            f"  Q   = {self.states}\n"
            f"  Σ   = {self.input_alphabet}\n"
            f"  Γ   = {self.stack_alphabet}\n"
            f"  q0  = {self.start_state}\n"
            f"  Z0  = {self.initial_stack}\n"
            f"  F   = {self.final_states}\n"
            f"  mode= {self.acceptance_mode}\n"
            f"  δ   = {list(self.delta.values())}\n"
            f")"
        )

## Parser
Reads raw text input and extracts all DPDA components into a dictionary.  
Transition format: `from_state  input  stack_top  to_state  push`  
Use `eps` for λ input or empty push.

In [4]:
class ParseError(Exception):
    pass
class DPDAParser:
    """
    Expected format:
        States: q0 q1 q2
        Input alphabet: a b
        Stack alphabet: Z A
        Start state: q0
        Initial stack symbol: Z
        Final states: q2
        Acceptance mode: final

        Number of transitions: 4
        q0 a Z q1 AZ
        q1 a A q1 AA
        q1 b A q2 eps
        q2 b A q2 eps
    """

    def parse(self, raw_input):
        lines = [
            line.strip()
            for line in raw_input.strip().splitlines()
            if line.strip() and not line.strip().startswith('#')
        ]

        states = self._parse_field(lines, "States:")
        input_alphabet= self._parse_field(lines, "Input alphabet:")
        stack_alphabet = self._parse_field(lines, "Stack alphabet:")
        start_state  = self._parse_single(lines, "Start state:")
        initial_stack = self._parse_single(lines, "Initial stack symbol:")
        final_states = self._parse_field(lines, "Final states:")
        acceptance_mode= self._parse_single(lines, "Acceptance mode:").lower()
        n_transitions= self._parse_count(lines, "Number of transitions:")
        transitions  = self._parse_transitions(lines, n_transitions)

        return {
            'states' : set(states),
            'input_alphabet': set(input_alphabet),
            'stack_alphabet': set(stack_alphabet),
            'start_state': start_state,
            'initial_stack' : initial_stack,
            'final_states' : set(final_states),
            'acceptance_mode': acceptance_mode,
            'transitions' : transitions,
        }

    def _parse_field(self, lines, prefix):
        for line in lines:
            if line.lower().startswith(prefix.lower()):
                value = line[len(prefix):].strip()
                if not value:
                    raise ParseError(f"'{prefix}' section is empty.")
                return value.split()
            
            
        raise ParseError(f"Missing required section: '{prefix}'")

    def _parse_single(self, lines, prefix):
        values = self._parse_field(lines, prefix)
        if len(values) != 1:
            raise ParseError(
                f"'{prefix}' must have exactly one value, got: {values}"
            )
        return values[0]

    def _parse_count(self, lines, prefix):
        values = self._parse_field(lines, prefix)
        if len(values) != 1 or not values[0].isdigit():
            raise ParseError(
                f"'{prefix}' must be a single non-negative integer, got: {values}"
            )
        return int(values[0])

    def _parse_transitions(self, lines, n):
        section_prefixes = (
            "states:", "input alphabet:", "stack alphabet:","start state:", "initial stack symbol:","final states:", "acceptance mode:", "number of transitions:",)
        transition_lines = [
            line for line in lines
            if not any(line.lower().startswith(p) for p in section_prefixes)
        ]

        if len(transition_lines)<n:
            raise ParseError(
                f"Expected {n} transition(s), but only found {len(transition_lines)}."
            )

        transitions = []
        for line in transition_lines[:n]:
            parts = line.split()
            if len(parts) != 5:
                raise ParseError(
                    f"Invalid transition format: '{line}'.\n"
                    f"  Expected: from_state  input  stack_top  to_state  push_string"
                )

            from_state, input_sym, stack_top, to_state, push_str = parts

            if input_sym.lower() in ('lambda', 'ε', 'λ','eps'):
                input_sym = 'eps'
            if push_str.lower() in ('lambda', 'ε', 'λ','eps'):
                push_str = 'eps'

            transitions.append({
                'from_state': from_state,
                'input'  : input_sym,
                'stack_top': stack_top,
                'to_state': to_state,
                'push': push_str,
            })

        return transitions

## Validator
Checks the parsed DPDA for correctness before simulation.

| Rule | Description |
|---|---|
| START STATE | Start state must be in Q |
| FINAL STATES | All final states must be in Q |
| INITIAL STACK | Z₀ must be in Γ |
| ACCEPTANCE MODE | Must be `final` or `empty` |
| TRANSITION FROM/TO | Both states must be in Q |
| TRANSITION INPUT | Symbol must be in Σ or `eps` |
| TRANSITION STACK TOP | Symbol must be in Γ |
| TRANSITION PUSH | All push symbols must be in Γ |
| DETERMINISM | At most one transition per (state, input, stack_top) |
| λ-CONFLICT | If δ(q, λ, X) exists, δ(q, a, X) must not exist for any a ∈ Σ |

In [5]:
from dataclasses import dataclass

@dataclass
class ValidationError:
    rule:str
    message:str

    def __str__(self):
        return f"[{self.rule}] {self.message}"


class DPDAValidator:

    def validate(self, dpda_dict):
        states= dpda_dict['states']
        input_alphabet= dpda_dict['input_alphabet']
        stack_alphabet= dpda_dict['stack_alphabet']
        start_state= dpda_dict['start_state']
        initial_stack= dpda_dict['initial_stack']
        final_states= dpda_dict['final_states']
        acceptance_mode= dpda_dict['acceptance_mode']
        transitions= dpda_dict['transitions']

        errors =[]

        self._check_start_state(start_state, states, errors)
        self._check_final_states(final_states, states, errors)
        self._check_initial_stack(initial_stack, stack_alphabet, errors)
        self._check_acceptance_mode(acceptance_mode, errors)
        self._check_transitions(transitions, states, input_alphabet, stack_alphabet, errors)
        self._check_determinism(transitions, input_alphabet, errors)
        return len(errors) == 0, errors


    def _check_start_state(self, start_state, states, errors):
        if start_state not in states:
            errors.append(ValidationError(rule="START STATE",message=f"Start state '{start_state}' is not in the states set {states}."))

    def _check_final_states(self, final_states, states, errors):
        for state in final_states:
            if state not in states:errors.append(ValidationError(rule="FINAL STATES",message=f"Final state '{state}' is not in the states set {states}."))

    def _check_initial_stack(self, initial_stack, stack_alphabet, errors):
        if initial_stack not in stack_alphabet:errors.append(ValidationError(rule="INITIAL STACK SYMBOL",message=f"Initial stack symbol '{initial_stack}' is not in the stack alphabet {stack_alphabet}."))

    def _check_acceptance_mode(self, acceptance_mode, errors):
        if acceptance_mode not in ('final', 'empty'):errors.append(ValidationError(rule="ACCEPTANCE MODE",message=f"Acceptance mode '{acceptance_mode}' is invalid. Must be 'final' or 'empty'."))

    def _check_transitions(self, transitions, states, input_alphabet, stack_alphabet, errors):
        for t in transitions:
            from_state=t['from_state']
            input_sym= t['input']
            stack_top   = t['stack_top']
            to_state    = t['to_state']
            push= t['push']

            if from_state not in states:
                errors.append(ValidationError(rule="TRANSITION FROM",message=f"Transition from unknown state '{from_state}' — not in states set."))

            if to_state not in states:
                errors.append(ValidationError(rule="TRANSITION TO",message=f"Transition to unknown state '{to_state}' — not in states set."))

            if input_sym != 'eps' and input_sym not in input_alphabet:
                errors.append(ValidationError(rule="TRANSITION INPUT",message=f"Input symbol '{input_sym}' in transition "f"({from_state}, {input_sym}, {stack_top}) is not in the input alphabet."))  

            if stack_top not in stack_alphabet:
                errors.append(ValidationError(rule="TRANSITION STACK TOP",message=f"Stack symbol '{stack_top}' in transition "f"({from_state}, {input_sym}, {stack_top}) is not in the stack alphabet."))

            if push != 'eps':
                for sym in push:         
                    if sym not in stack_alphabet:
                        errors.append(ValidationError(rule="TRANSITION PUSH",message=f"Push symbol '{sym}' in transition "f"({from_state}, {input_sym}, {stack_top}) → push '{push}' "f"is not in the stack alphabet."))

    def _check_determinism(self, transitions, input_alphabet, errors):
        seen = set()
        for t in transitions:
            key = (t['from_state'], t['input'], t['stack_top'])
            if key in seen:
                errors.append(ValidationError(rule="DETERMINISM",message=f"Duplicate transition for "f"('{t['from_state']}', '{t['input']}', '{t['stack_top']}') — "f"at most one transition per (state, input, stack_top) is allowed."))
            else:
                seen.add(key)

        lambda_keys = {
            (t['from_state'], t['stack_top'])
            for t in transitions if t['input'] == 'eps'
        }
        for t in transitions:
            if t['input']!='eps':
                pair = (t['from_state'], t['stack_top'])
                if pair in lambda_keys:
                    errors.append(ValidationError(rule="DETERMINISM (λ-CONFLICT)",message=f"Transition ({t['from_state']}, '{t['input']}', '{t['stack_top']}') "f"conflicts with an existing λ-transition on the same "f"(state, stack_top) pair — this makes the machine non-deterministic."))

    def _report_errors(self, errors):
        print("\n  Invalid DPDA — the following error(s) were found:\n")
        for i, error in enumerate(errors, start=1):
            print(f"  {i}. {error}")
        print("\n   Please fix the above and try again.\n")

## Simulator
Drives the step-by-step execution of the DPDA on an input string.

Each step prints:
- Symbol read from input (or `λ` for epsilon moves)
- Stack operation performed
- Current state and stack contents after the transition

Acceptance is evaluated based on the mode specified in the input:
- **Final state** — input consumed and current state ∈ F
- **Empty stack** — input consumed and stack is empty

In [7]:


GREEN = "\033[92m"
RED   = "\033[91m"
RESET = "\033[0m"

class DPDASimulator:
    
    def run(self, dpda, input_string):
        display_string = input_string if input_string else '(empty)'
        print(f"\nInput string: {display_string}")
        print(f"Acceptance mode: {dpda.acceptance_mode}")
        print()

        current_state = dpda.start_state
        stack = dpda.fresh_stack()
        index = 0 

        print(f"State: {current_state} , Stack: {stack}")

        while True:

            if index < len(input_string):
                current_input = input_string[index]
            
            else:
            
                current_input = 'eps'

            if stack.is_empty():
                break

            stack_top = stack.top()


            transition = dpda.get_transition(current_state, current_input, stack_top)
            if transition is None:
                break

            action  = dpda.describe_action(transition)
            stack  = dpda.apply_transition(stack, transition)
            next_state= transition.to_state

            if transition.input != 'eps':
                symbol_read = current_input
                index += 1
                
            else:
                symbol_read = 'λ'
                
            print(f"Read {symbol_read} -> {action}")
            print(f"State: {next_state} , Stack: {stack}")
            print()

            current_state = next_state

        input_consumed = (index == len(input_string))
        
        if dpda.acceptance_mode == 'final':
            accepted = input_consumed and dpda.is_accepting_final(current_state)
        else:
            accepted = input_consumed and dpda.is_accepting_empty(stack)

        print(f"Halted at state: {current_state}")
        print(f"Acceptance mode: {dpda.acceptance_mode}")
        if accepted:
            print(f"{GREEN}Result: Accepted{RESET}")
        else:
            print(f"{RED}Result: Rejected{RESET}")
        print()

        return accepted

## Build Helper
Wires together the parser, validator, and DPDA constructor.  
Returns a ready-to-use `DPDA` object, or `None` if validation fails.

In [8]:
def build_dpda(raw_input):
    parser= DPDAParser()
    validator = DPDAValidator()
    dpda_dict = parser.parse(raw_input)
    is_valid, errors = validator.validate(dpda_dict)

    if not is_valid:
        validator._report_errors(errors)
        return None

    transitions = [
        DPDATransition(
            from_state = t['from_state'],
            input= t['input'],
            stack_top  = t['stack_top'],
            to_state = t['to_state'],
            push  = t['push'],
        )
        for t in dpda_dict['transitions']
    ]
    return DPDA(dpda_dict, transitions)

## Input Collector
Prompts the user field by field so they never have to type prefixes manually.

In [9]:
def collect_input():
    lines = []

    prompts = [
        ("States","  States: "),
        ("Input alphabet", "  Input alphabet: "),
        ("Stack alphabet", "  Stack alphabet: "),
        ("Start state", "  Start state: "),
        ("Initial stack symbol", "  Initial stack symbol: "),
        ("Final states",   "  Final states: "),
        ("Acceptance mode",     "  Acceptance mode (final / empty): "),
    ]

    for prefix, prompt in prompts:
        while True:
            try:
                value = input(prompt).strip()
            except EOFError:
                return ""
            if value:
                lines.append(f"{prefix}: {value}")
                break
            print("    This field cannot be empty. Please try again.")

    while True:
        try:
            n_str = input("  Number of transitions: ").strip()
        except EOFError:
            return ""
        if n_str.isdigit():
            n = int(n_str)
            lines.append(f"Number of transitions: {n_str}")
            break
        print("    Please enter a valid non-negative integer.")

    print("  Enter each transition as:  from_state  input  stack_top  to_state  push")
    print("  (use 'eps' for lambda input or empty push)\n")

    for i in range(n):
        while True:
            try:
                t = input(f"  Transition {i + 1}: ").strip()
            except EOFError:
                return ""
            if len(t.split()) == 5:
                lines.append(t)
                break
            print("     Expected exactly 5 fields. Try again.")

    return "\n".join(lines)

## Run
Enter your DPDA definition below and test strings interactively.  
Type `exit` to stop testing.

> **Note:** Re-run this cell each time you want to define a new DPDA.

In [ ]:
simulator = DPDASimulator()
raw_input = collect_input()

if raw_input.strip():
    dpda = build_dpda(raw_input)
    if dpda:
        print(dpda)
        while True:
            user_input = input("\n  String to test or exit to stop: ").strip()
            if user_input.lower() == "exit":
                break
            simulator.run(dpda, user_input)